## <font color="royalblue">**Extracción de datos**</font>  
### <font color="royalblue">**API Adzuna**</font>
En estes archivo se extraen los datos del portal web Adzuna para ofertas de empleo en Data Analyst para Barcelona.
- Adzuna ha creado una API RESTful (https://developer.adzuna.com/) para obtener datos y anuncios de empleo del portal web Adzuna.
- Registrando un usuario en API Adzuna se genera una cuenta para acceder a ID y KEY, los mismos permiten extraer las ofertas de empleo.
    - url base: "https://api.adzuna.com/v1/api/jobs/{country}/search/{page}"
    - url con parametros: http://api.adzuna.com/v1/api/jobs/es/search/1?app_id={API ID}&app_key={API KEY}&results_per_page=20&what=datat%20analisis&content-type=application/json

- Se crea la función *buscar_ofertas_adzuna* para gestionar url, parametros y respuesta de la petición.  
- En resultados se almacena los datos de la petición, recorriendo con un **for** para extraer las variables requeridas.
- La variables son cargadas en un dataframe *df_Adzuna* y exportados como un archivo .csv. 

In [ ]:
import requests
import pandas as pd

APP_ID = "9361dc98"                          
APP_KEY = "0888d8a507dfcefb7d2e5887141a5a63" 

In [ ]:
# Función para buscar ofertas
def buscar_ofertas_adzuna(what, where, page=1, results=20):
    """función para obtener resultados de la request a la pagina API Adzuna"""
    url = f"https://api.adzuna.com/v1/api/jobs/es/search/{page}"

    params = {
        "app_id": APP_ID,
        "app_key": APP_KEY,
        "what": what,
        "where": where,
        "results_per_page": results,
        "content-type": "application/json"
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        print("Error:", response.status_code, response.text)
        return None

    data = response.json()
    return data

In [ ]:
datos = []

for page in range(1, 9):  # páginas 1 a 5
    resultados = buscar_ofertas_adzuna("data analyst", "Barcelona", page=page)

    for job in resultados["results"]:
        datos.append({
        "title": job.get("title"),
        "company": job.get("company", {}).get("display_name"),
        "location": job.get("location", {}).get("display_name"),
        "contract_type": job.get("contract_type"),
        "contract_time": job.get("contract_time"),
        "description": job.get("description"),
        "category": job.get("category", {}).get("label"),
        "url_": job.get("redirect_url"),
        "id": job.get("id"),
        "date": job.get("created")
        })

df_Adzuna = pd.DataFrame(datos)


,title,company,location,contract_type,contract_time,description,category,url_,id,date
0,Data Analyst,Perk,Barcelona,None,None,About Us Perk (formerly TravelPerk) is the int...,Unknown,https://www.adzuna.es/details/5655420542?utm_m...,5655420542,2026-03-05T22:17:29Z
1,Data Analyst,Veepee,Barcelona,None,None,Pioneer of online flash sales since 2001 and k...,Unknown,https://www.adzuna.es/details/5590818342?utm_m...,5590818342,2026-01-18T22:18:49Z
2,Data Analyst,Propelling Tech,Barcelona,None,None,"As a Data Analyst , you will be part of a high...",Unknown,https://www.adzuna.es/details/5584544401?utm_m...,5584544401,2026-01-14T20:09:21Z
3,Data Analyst,PrimeIT,Barcelona,None,None,Qué buscamos? A partir de 3 años de experienci...,Unknown,https://www.adzuna.es/details/5304781614?utm_m...,5304781614,2025-07-16T00:10:07Z
4,Data Analyst Jr,Solicitud de empleo para Data Analyst Jr en DD...,Barcelona,None,None,"Perfil Si te apasionan los datos, quieres apre...",Unknown,https://www.adzuna.es/details/5556867176?utm_m...,5556867176,2025-12-26T20:51:02Z


In [4]:
df_Adzuna.to_csv("df_Adzuna.csv", index=False)